# Submit nudged elastic band calculation

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# General imports.
import ipywidgets as ipw
from IPython.display import clear_output

# AiiDA imports.
%load_ext aiida
%aiida

# AiiDAlab imports.
import aiidalab_widgets_base as awb
import aiidalab_widgets_empa as awe
from aiida import orm, plugins

# Custom imports.
from surfaces_tools.widgets import editors, inputs

In [ ]:
Cp2kNebWorkChain = plugins.WorkflowFactory("nanotech_empa.cp2k.neb")

In [ ]:
# Structure selector.
build_slab = editors.BuildSlab(title="Build slab")
input_details = inputs.InputDetails()

structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
        awe.CdxmlUploadWidget(title="CDXML"),
    ],
    editors=[awb.BasicStructureEditor(title="Edit structure"), build_slab],
    storable=False,
    node_class="StructureData",
)

ipw.dlink((structure_selector, "structure"), (build_slab, "molecule"))
ipw.dlink((structure_selector, "structure"), (input_details, "structure"))
ipw.dlink((input_details, "details"), (build_slab, "details"))
input_details.neb = True
input_details.replica = False

display(structure_selector)

# Code.
code_input_widget = awb.ComputationalResourcesWidget(
    description="CP2K code:", default_calc_job_plugin="cp2k"
)
resources = awe.ProcessResourcesWidget()

# Protocol.
protocol = ipw.Dropdown(
    value="standard",
    options=[
        ("Standard", "standard"),
        ("Low accuracy", "low_accuracy"),
        ("Debug", "debug"),
    ],
    description="Protocol:",
    style={"description_width": "initial"},
)

output = ipw.Output()

In [ ]:
workflow_description = ipw.Text(
    description="Workflow description:",
    placeholder="Provide the description here.",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

In [ ]:
ipw.dlink((code_input_widget, "value"), (input_details, "selected_code"))


def prepare_inputs():
    with output:
        clear_output()
    if not structure_selector.structure_node:
        can_submit, msg = False, "Select a structure first."
    elif not code_input_widget.value:
        can_submit, msg = False, "Select CP2K code."
    else:
        can_submit, msg, parameters = input_details.return_final_dictionary()

    if not can_submit:
        with output:
            print(msg)
            return

    builder = Cp2kNebWorkChain.get_builder()
    builder.protocol = orm.Str(protocol.value)
    builder.metadata.label = "CP2K_NEB"
    builder.metadata.description = workflow_description.value
    builder.code = orm.load_code(code_input_widget.value)
    builder.options = {
        "max_wallclock_seconds": resources.walltime_seconds,
        "resources": {
            "num_machines": resources.nodes,
            "num_mpiprocs_per_machine": resources.tasks_per_node,
            "num_cores_per_mpiproc": resources.threads_per_task,
        },
    }

    builder.structure = structure_selector.structure_node
    builder.dft_params = orm.Dict(parameters["dft_params"])
    builder.sys_params = orm.Dict(parameters["sys_params"])
    builder.neb_params = orm.Dict(parameters["neb_params"])

    if "replica_uuids" in parameters:
        replicas = {}
        for i in range(len(parameters["replica_uuids"])):
            name = "replica_" + str(i + 1).zfill(3)
            replicas[name] = orm.load_node(parameters["replica_uuids"][i])
        builder.replicas = replicas

    if "restart_from" in parameters:
        builder.restart_from = orm.Str(parameters["restart_from"])

    return builder

In [ ]:
btn_submit = awb.SubmitButtonWidget(
    Cp2kNebWorkChain,
    inputs_generator=prepare_inputs,
    disable_after_submit=False,
    append_output=True,
)

In [ ]:
# Resources estimation.
resources_estimation_button = awe.ResourcesEstimatorWidget(
    price_link="https://2go.cscs.ch/offering/swiss_academia/institutional_customers/",
    price_per_hour=2.85,
)
resources_estimation_button.link_to_resources_widget(resources)
ipw.dlink((input_details, "details"), (resources_estimation_button, "details"))
ipw.dlink((input_details, "uks"), (resources_estimation_button, "uks"))
ipw.dlink((input_details, "neb"), (resources, "neb"))
ipw.dlink((input_details, "n_replica_trait"), (resources, "n_replica_trait"))
ipw.dlink(
    (input_details, "n_replica_per_group_trait"),
    (resources, "n_replica_per_group_trait"),
)
ipw.dlink((resources, "nproc_replica_trait"), (input_details, "nproc_replica_trait"))
_ = ipw.dlink(
    (code_input_widget, "value"), (resources_estimation_button, "selected_code")
)

# Inputs

In [ ]:
display(protocol, input_details)

# Code and resources

In [ ]:
display(code_input_widget, ipw.HBox([resources, resources_estimation_button]))

# Submit

In [ ]:
display(workflow_description, btn_submit, output)